In [170]:
import numpy as np

known = dict()
def min_pins_for_100(score=0, npins=0, bonus=None, frame=1):
    if bonus is None:
        bonus = np.array([0, 0])
    dkey = '%03d|%03d|%1d|%1d|%02d' % (score, npins, *bonus, frame)
    if dkey not in known:
        if frame == 10:
            sc_poss = np.array([[i, j, k] 
                                for i in range(11) 
                                for j in range(11-(i % 10)) 
                                for k in range(11-(j % 10) if i == 10 else 11 if i+j == 10 else 1)])
            new_score = score + np.sum(sc_poss, axis=1) + np.sum(sc_poss[:, :2] * bonus, axis=1)
            new_npins = npins + np.sum(sc_poss, axis=1)
            if np.sum(new_score >= 100) == 0:
                known[dkey] = 9999
            else:
                known[dkey] = int(np.min(new_npins[new_score >= 100]))
        else:
            sc_poss = np.array([[i, j] 
                                for i in range(11) 
                                for j in range(11-i)])
            new_score = score + np.sum(sc_poss, axis=1) + np.sum(sc_poss * bonus, axis=1)
            new_npins = npins + np.sum(sc_poss, axis=1)
            new_bonus = np.where(sc_poss[:, :1] == 10, 
                                 np.append(bonus[1:], [0]) + [1, 1], 
                                 np.where(np.sum(sc_poss, axis=1, keepdims=True) == 10,
                                          [1, 0],
                                          [0, 0]))
            known[dkey] = int(np.min([new_npins[i] 
                                      if new_score[i] >= 100 
                                      else min_pins_for_100(new_score[i], 
                                                            new_npins[i], 
                                                            new_bonus[i], 
                                                            frame+1)
                                      for i in range(sc_poss.shape[0])]))

    return known[dkey]

min_pins_for_100()

44

In [384]:
known = dict()
def score_range_by_npins(bonus=None, frame=1):
    if bonus is None:
        bonus = np.array([0, 0])
    dkey = '%1d|%1d|%02d' % (*bonus, frame)
    if dkey not in known:
        if frame == 10:
            sc_poss = np.array([[i, j, k] 
                                for i in range(11) 
                                for j in range(11-(i % 10)) 
                                for k in range(11-(j % 10) if i == 10 else 11 if i+j == 10 else 1)])
            inc_score = np.sum(sc_poss, axis=1) + np.sum(sc_poss[:, :2] * bonus, axis=1)
            inc_npins = np.sum(sc_poss, axis=1)
            known[dkey] = {int(i): [int(np.min(inc_score[inc_npins == i])), 
                                    int(np.max(inc_score[inc_npins == i]))]
                           for i in np.unique(inc_npins)}
        else:
            sc_poss = np.array([[i, j] 
                                for i in range(11) 
                                for j in range(11-i)])
            inc_score = np.sum(sc_poss, axis=1) + np.sum(sc_poss * bonus, axis=1)
            inc_npins = np.sum(sc_poss, axis=1)
            new_bonus = np.where(sc_poss[:, :1] == 10, 
                                 np.append(bonus[1:], [0]) + [1, 1], 
                                 np.where(np.sum(sc_poss, axis=1, keepdims=True) == 10,
                                          [1, 0],
                                          [0, 0]))
            
            known[dkey] = dict()
            for i in np.unique(inc_npins):
                for j in np.arange(sc_poss.shape[0])[inc_npins == i]:
                    dsdict = score_range_by_npins(new_bonus[j], frame+1)
                    for k in dsdict:
                        new_npins = int(i+k)
                        lsc = int(np.sum(sc_poss[j] * (bonus+1))+dsdict[k][0])
                        hsc = int(np.sum(sc_poss[j] * (bonus+1))+dsdict[k][1])
                        if new_npins not in known[dkey]:
                            known[dkey][new_npins] = [lsc, hsc]
                        else:
                            if lsc < known[dkey][new_npins][0]:
                                known[dkey][new_npins][0] = lsc
                            if hsc > known[dkey][new_npins][1]:
                                known[dkey][new_npins][1] = hsc

    return known[dkey]

rdict = score_range_by_npins()
tot_pins = np.sort(list(rdict.keys()))
diffs = np.array([rdict[k][1]-rdict[k][0] for k in tot_pins])

int(np.max(diffs))

180